# 01 · 类型与接口实验

目标：亲手验证 struct 值语义、隐式接口和 typed nil。对应文档：`01_language_basics/1.1-types-and-interfaces.md`。

## 从 TypeScript 迁移

Go struct 赋值默认复制值，不像 JS 对象默认复制引用。接口由方法集合隐式满足，接近结构化类型；但接口值同时保存动态类型和值，因此 typed nil 是新的陷阱。

In [ ]:
import "fmt"

type User struct { Name string }

## 实验一：值语义

先预测：修改 `b.Name` 后，`a.Name` 会不会一起变化？然后运行下面的 cell。

In [ ]:
%%
a := User{Name: "Ada"}
b := a
b.Name = "Grace"
fmt.Printf("a=%s b=%s\n", a.Name, b.Name)

结果应为 `a=Ada b=Grace`。赋值复制了整个 struct。若字段是 slice、map 或 pointer，复制的是描述符或地址，底层数据仍可能共享。

In [ ]:
type Loader interface {
    Load(id string) ([]byte, error)
}

type MemoryLoader map[string][]byte

func (m MemoryLoader) Load(id string) ([]byte, error) {
    value, ok := m[id]
    if !ok { return nil, fmt.Errorf("%s not found", id) }
    return value, nil
}

var _ Loader = MemoryLoader{}

## 实验二：隐式接口

`MemoryLoader` 没有声明 implements。编译期断言 `var _ Loader = MemoryLoader{}` 证明它的方法集合满足 `Loader`。

In [ ]:
%%
loader := MemoryLoader{"intro": []byte("Go")}
value, err := loader.Load("intro")
fmt.Printf("value=%s err=%v\n", value, err)

In [ ]:
type Problem struct{}
func (*Problem) Error() string { return "problem" }

## 实验三：typed nil

先预测 `err == nil`。接口里装入 `(*Problem)(nil)` 后，动态类型仍是 `*Problem`，所以接口本身不为 nil。

In [ ]:
%%
pointer := (*Problem)(nil)
err := error(pointer)
fmt.Printf("pointer == nil: %t, interface == nil: %t\n", pointer == nil, err == nil)

## 动手挑战

1. 给 `User` 增加 `Tags []string`，复制后修改 `b.Tags[0]`，观察共享底层数组。
2. 把 `Load` 改成指针接收者，观察 `MemoryLoader{}` 的编译期断言如何变化。

## 复盘

- struct 复制为什么不等于深拷贝？
- 接口应该定义在实现方还是使用方？
- 如何避免返回含 typed nil 的 `error`？